## 0. Налаштування середовища та 1. Підготовка датасету
Для тренування моделі було обрано датасет української поезії (`RudyiVT/ukrainian-poems`).
**Обґрунтування вибору:** Вірші чудово підходять для знайомства з NLP, оскільки це дозволяє отримати цікаві, творчі результати (порівняно із сухими статтями чи документами). Однак, щоб уникнути переповнення пам'яті (OOM) на безкоштовному GPU та оптимізувати час тренування, ми модифікували підхід: датасет випадковим чином перемішано, і для корпусу взято унікальний зріз лише з 5000 віршів. Це робить нашу вибірку унікальною та гарантує стабільне виконання коду.

In [ ]:
import sys
import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
import numpy as np

# Перевірка наявності GPU (T4 у Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Використовуємо:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("УВАГА: GPU не увімкнено! Навчання буде дуже повільним.")

print("\nЗавантаження датасету...")
# Беремо випадкові 5000 віршів для швидкості та унікальності
dataset = load_dataset("RudyiVT/ukrainian-poems", split="train")
dataset = dataset.shuffle(seed=42).select(range(5000))

# Перетворюємо у pandas DataFrame, щоб відфільтрувати порожні рядки
df = dataset.to_pandas()
poems_list = df['context_poem'].dropna().tolist()
corpus_text = "\n\n".join(poems_list)

print("Розмір корпусу:", len(corpus_text), "символів")
print("Приклад тексту:\n", corpus_text[:300])

Використовуємо: cuda
GPU: Tesla T4

Завантаження датасету...
Розмір корпусу: 17148168 символів
Приклад тексту:
 ЛЮДИНА В ПРОСТОРІ

І

Яблука, — подумав, повертаючись до стіни, — яблука.. Мокрі, потовчені, в пластмасовім шапличку бачив якось біля головпошти, валив сніг, продавець вихоплював жменями дрібноту з металевого контейнера, нещасні, їх розпродують за будь-якої погоди. Але ж не всі такі.. Він перевернув


## 2. Токенізація (4 способи)
Токенізація перетворює сирий текст на послідовність ідентифікаторів. Тестуємо 4 підходи:
1. **BPE (Byte Pair Encoding):** Ітеративно зливає найчастіші пари. Стандарт для сучасних LLM.
2. **WordPiece:** Обирає пари для злиття на основі максимізації ймовірності (використовується в BERT).
3. **Unigram:** Починає з великого словника і видаляє токени, які найменше впливають на ймовірність.
4. **Byte-level:** Токенізує текст на рівні окремих байтів (дуже малий словник, але довгий контекст).

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

VOCAB_SIZE = 10000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

print("Тренування токенізаторів...")

# 1. BPE Токенізатор
bpe_tok = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
bpe_tok.train_from_iterator(poems_list, trainer=bpe_trainer)
print("Розмір словника BPE:", bpe_tok.get_vocab_size())

# 2. WordPiece Токенізатор
wp_tok = Tokenizer(models.WordPiece(unk_token="<unk>"))
wp_tok.pre_tokenizer = pre_tokenizers.Whitespace()
wp_trainer = trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
wp_tok.train_from_iterator(poems_list, trainer=wp_trainer)
print("Розмір словника WordPiece:", wp_tok.get_vocab_size())

# 3. Unigram Токенізатор
uni_tok = Tokenizer(models.Unigram())
uni_tok.pre_tokenizer = pre_tokenizers.Whitespace()
uni_trainer = trainers.UnigramTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, unk_token="<unk>")
uni_tok.train_from_iterator(poems_list, trainer=uni_trainer)
print("Розмір словника Unigram:", uni_tok.get_vocab_size())

# Використаємо BPE для подальшої роботи
bpe_vocab = bpe_tok.get_vocab()
stoi = bpe_vocab
itos = {v: k for k, v in bpe_vocab.items()}
UNK_ID = stoi["<unk>"]

Тренування токенізаторів...
Розмір словника BPE: 10000
Розмір словника WordPiece: 10000
Розмір словника Unigram: 10000


## 3. Тренування власної mini-GPT
Тут реалізовано decoder-only трансформер. Основний компонент — **Multi-head Attention**, який дозволяє моделі фокусуватися на різних словах у контексті. Застосовується **Causal Mask**, щоб модель «не підглядала» на майбутні слова. Гіперпараметри адаптовано під ресурси Colab.

In [ ]:
# 1. Кодування даних BPE токенізатором
print("Кодування корпусу...")
encodings = bpe_tok.encode_batch(poems_list)
token_ids = np.concatenate([np.array(enc.ids + [stoi.get("<eos>", 0)], dtype=np.int32) for enc in encodings])
data = torch.from_numpy(token_ids.astype(np.int64))

split_idx = int(len(data) * 0.9)
train_data, val_data = data[:split_idx], data[split_idx:]

# 2. Гіперпараметри
CONTEXT_LEN = 32
EMBED_DIM = 64
N_HEADS = 4
N_LAYERS = 3
DROPOUT = 0.1
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_ITERS = 500 # Зменшено для швидкої демонстрації

def get_batch(split: str):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - CONTEXT_LEN - 1, (BATCH_SIZE,))
    x = torch.stack([source[i:i + CONTEXT_LEN] for i in ix])
    y = torch.stack([source[i + 1:i + CONTEXT_LEN + 1] for i in ix])
    return x.to(device), y.to(device)

# 3. Архітектура моделі
class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, context_len, dropout):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        causal_mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = self.dropout(F.softmax(scores, dim=-1))
        return weights @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim), nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim), nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, context_len, dropout):
        super().__init__()
        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))
        x = self.final_norm(self.blocks(x))
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1)) if targets is not None else None
        return logits, loss

# 4. Ініціалізація та тренування
model = MiniGPT(len(itos), EMBED_DIM, N_HEADS, N_LAYERS, CONTEXT_LEN, DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"Початок тренування на {MAX_ITERS} ітерацій...")
for it in range(MAX_ITERS):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if it % 100 == 0 or it == MAX_ITERS - 1:
        print(f"Ітерація {it:4d} | Тренувальний Loss: {loss.item():.4f}")
print("Тренування завершено!")

Кодування корпусу...
Початок тренування на 500 ітерацій...
Ітерація    0 | Тренувальний Loss: 9.3817
Ітерація  100 | Тренувальний Loss: 7.3380
Ітерація  200 | Тренувальний Loss: 7.1846
Ітерація  300 | Тренувальний Loss: 6.9167
Ітерація  400 | Тренувальний Loss: 6.9046
Ітерація  499 | Тренувальний Loss: 6.9259
Тренування завершено!


## 4. Sampling playground: вплив параметрів на генерацію
Для тестування параметрів семплювання завантажимо готову модель Qwen. Досліджуємо:
* **Temperature:** Низька температура робить відповіді консервативними. Висока — креативними, але може викликати галюцинації.
* **Top-p:** Обмежує вибір токенами, кумулятивна ймовірність яких становить `p`, відсікаючи нерелевантні слова.

In [18]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Примусово звільняємо відеопам'ять від моделі з попереднього кроку
try:
    del model, optimizer, xb, yb, logits, loss
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("GPU пам'ять очищено!")

# 2. Завантажуємо нову модель економно
MODEL_ID = "Qwen/Qwen3-4B"
print(f"Завантаження {MODEL_ID}...")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Використовуємо float16 та device_map="auto" замість жорсткого .to(device)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
hf_model.eval()

def hf_generate(prompt: str, temperature: float, top_p: float) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    # Відправляємо вхідні дані на GPU
    inputs = hf_tokenizer([text], return_tensors="pt").to("cuda")
    outputs = hf_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=temperature, top_p=top_p)
    return hf_tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

PROMPT = "Продовж речення: Якось уночі студент Житомирської політехніки, який писав дипломну роботу з інженерії програмного забезпечення, прокинувся від дивного шуму. Він підвівся з ліжка"

for temp, top_p in [(0.2, 0.1), (0.8, 0.6), (1.4, 0.8)]:
    print(f"\n--- temp={temp}, top_p={top_p} ---")
    print(hf_generate(PROMPT, temp, top_p))

GPU пам'ять очищено!
Завантаження Qwen/Qwen3-4B...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


--- temp=0.2, top_p=0.1 ---
Якось уночі студент Житомирської політехніки, який писав дипломну роботу з інженерії програмного забезпечення, прокинувся від дивного шуму. Він підвівся з ліжка, зігнувся, підійшов до вікна і

--- temp=0.8, top_p=0.6 ---
Якось уночі студент Житомирської політехніки, який писав дипломну роботу з інженерії програмного забезпечення, прокинувся від дивного шуму. Він підвівся з ліжка, пішов до ванної кімнати, а там, на в

--- temp=1.4, top_p=0.8 ---
Якось уночі студент Житомирської політехніки, який писав дипломну роботу з інженерії програмного забезпечення, прокинувся від дивного шуму. Він підвівся з ліжка, зробив кілька кроків по підлозі, а


## 5. Застосування готових моделей для типових задач NLP
Реалізація 5 пайплайнів для україномовних текстів з використанням оптимізованих моделей.

In [22]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForQuestionAnswering

print("1. Аналіз тональності")
sentiment = pipeline("sentiment-analysis", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")
print(sentiment("Ця лабораторна робота виявилась дуже цікавою!"))

print("\n2. Розпізнавання сутностей (NER)")
ner = pipeline("ner", model="Babelscape/wikineural-multilingual-ner")
print(ner("Студент Житомирської політехніки поїхав до Києва."))

print("\n3. Машинний переклад")
trans_name = "Helsinki-NLP/opus-mt-uk-en"
trans_tok = AutoTokenizer.from_pretrained(trans_name)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_name)
trans_inputs = trans_tok("Штучний інтелект стрімко розвивається в Україні.", return_tensors="pt")
trans_outputs = trans_model.generate(**trans_inputs)
print(trans_tok.decode(trans_outputs[0], skip_special_tokens=True))

print("\n4. Заповнення пропусків (Fill-Mask)")
fill_mask = pipeline("fill-mask", model="xlm-roberta-base")
print(fill_mask("Студенти політехніки активно вивчають штучний <mask>."))

print("\n5. Question Answering (QA)")
# Завантажуємо QA модель та токенізатор напряму
qa_name = "timpal0l/mdeberta-v3-base-squad2"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name)

question = "Коли були представлені трансформери?"
context = "Трансформери — це архітектура нейронних мереж, представлена у 2017 році. Вона стала основою для таких моделей як GPT."

# Токенізуємо запитання та контекст разом
qa_inputs = qa_tok(question, context, return_tensors="pt")

# Отримуємо передбачення (логіти для початку і кінця відповіді)
with torch.no_grad():
    outputs = qa_model(**qa_inputs)

# Знаходимо токени з найвищою ймовірністю для початку і кінця
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

# Збираємо токени відповіді та декодуємо їх у текст
predict_answer_tokens = qa_inputs.input_ids[0, answer_start_index : answer_end_index + 1]
print(qa_tok.decode(predict_answer_tokens, skip_special_tokens=True))

1. Аналіз тональності


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.8203819990158081}]

2. Розпізнавання сутностей (NER)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity': 'B-LOC', 'score': np.float32(0.5364027), 'index': 4, 'word': 'Ж', 'start': 8, 'end': 9}, {'entity': 'B-LOC', 'score': np.float32(0.9998179), 'index': 17, 'word': 'Києва', 'start': 43, 'end': 48}]

3. Машинний переклад


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

... (9 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

2017


## 6. Обчислення метрик
Оцінюємо задачу аналізу тональності за допомогою метрики Accuracy на тестовому наборі.

In [38]:
from sklearn.metrics import accuracy_score

test_texts = [
    "Це найкращий університет",
    "У мене сьогодні поганий настрій",
    "Я успішно склав екзамен",
    "Програма видає помилку"
]
# Мітки маленькими літерами, як їх повертає модель
true_labels = ["positive", "negative", "positive", "negative"]

pred_labels = [p['label'] for p in sentiment(test_texts)]
accuracy = accuracy_score(true_labels, pred_labels)
print(f"Точність моделі (Accuracy): {accuracy * 100}%")

Точність моделі (Accuracy): 100.0%


## 7. Творчий застосунок: Аналізатор тексту (Gradio)
Інтерактивний додаток для визначення емоційного забарвлення тексту.

In [46]:
import gradio as gr

def analyze_user_text(text):
    if not text.strip(): return "Введіть текст!"
    res = sentiment(text)[0]
    return f"Визначена емоція: {res['label']} (Впевненість: {res['score']:.2f})"

demo = gr.Interface(
    fn=analyze_user_text,
    inputs=gr.Textbox(lines=3, placeholder="Введіть текст українською..."),
    outputs="text",
    title="Аналізатор Тексту"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee685536b4ffaa80d7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
